# Skill 41: Quantization Foundations: FP32 to INT8

**Phase 3 - LLMs & Generative AI Applications (Skills 31-45)**

## What this skill is about
Map floating-point tensors to signed 8-bit integers with a symmetric scale, recover them, and bound the error.

## Objectives
- Calculate a symmetric scale from maximum absolute magnitude.
- Round and clamp quantized values to [-127, 127].
- Recover approximate floating-point values and measure quantization error.

Run the cells top to bottom. Requires `torch`, `numpy` and `matplotlib` (`pip install torch numpy matplotlib`).

## Theory and core formulas
$$s=\frac{\max|x|}{127},\quad q=\mathrm{clamp}(\mathrm{round}(x/s),-127,127),\quad \hat x=qs,\quad |x-\hat x|\le s/2$$

## Implementation
Reference code from the lab manual (unchanged).

In [ ]:
import torch

def quantize_symmetric_int8(x):
    max_abs = x.abs().max()
    scale = max_abs / 127.0
    scale = torch.where(scale == 0, torch.tensor(1.0, device=x.device), scale)
    q = torch.round(x / scale).clamp(-127, 127).to(torch.int8)
    return q, scale

def dequantize_int8(q, scale):
    return q.float() * scale

x = torch.tensor([-2.7, -1.1, -0.2, 0.0, 0.8, 2.4], dtype=torch.float32)
q, scale = quantize_symmetric_int8(x)
x_hat = dequantize_int8(q, scale)
print("original:", x)
print("int8:", q)
print("scale:", scale.item())
print("reconstructed:", x_hat)
print("max abs error:", (x - x_hat).abs().max().item())


## What to check
**Expected result:** `int8` tensor; max error about 0.0087 (below s/2 = 0.0106) for the sample tensor.

## Pitfalls
Do integer arithmetic in INT32 to avoid overflow. One outlier inflates the scale (use per-channel/group scales). This is simulated quantization, not a deployment runtime.